# Validating the new cross-match catalogs

This branch adds eight new counterpart catalogs to the astromon cross-match pipeline:
**RFC**, **ICRF3**, **GaiaAGN**, **GaiaQSO**, **Quaia**, **MilliquasGaia**, **DESIV161**, and
**GaiaVarStar** (see `docs/astromon.rst` for what each one is and why it was added).

The goal here is to check, catalog by catalog, that the matches the pipeline actually produces
look like *good* astrometric counterparts — i.e. counterparts whose positions can be trusted to
anchor the absolute astrometry / calibration solution — rather than noise that happened to fall
within the match radius.

For each new catalog, `astromon.cross_match.CROSS_MATCHES_ARGS` already defines a single-catalog
selection (`"rfc"`, `"icrf3"`, `"gaia_agn"`, `"gaia_qso"`, `"quaia"`, `"milliquas_gaia"`,
`"desi_v161"`, `"gaia_var_star"`) purpose-built for this kind of per-catalog diagnostic. We use
the pre-existing `"tycho2"` selection as a familiar baseline for comparison.

## Data

This notebook does **not** query the production DB directly. Instead it reads two small,
already-extracted sample files -- see `extract_catalog_validation_sample.py` in this directory:

- `catalog_match_validation_sample.ecsv` (~1 MB): a capped random sample of the precomputed
  matches for these nine selections. Each `astromon_xcorr` row for these selections is already
  filtered (dr, SNR, off-axis angle, near-neighbor-distance cuts baked in at the time the DB was
  populated — see `CROSS_MATCHES_ARGS`), so the sample only caps the row count per catalog (400
  max; GaiaVarStar has only 96 matches total); it does not re-filter. It also carries
  `dy_rebased`/`dz_rebased` -- `dy`/`dz` rebased to the latest CALALIGN alignment epoch (see
  section 4 below for why and how).
- `catalog_match_paired_sample.ecsv`: for each catalog, up to 400 counterparts that were matched
  independently by both a celldetect and a gaussian_detect X-ray source in the same obsid, used
  in section 6 to compare the two detection methods directly.

Set `ASTROMON_VALIDATION_SAMPLE_URL` / `ASTROMON_VALIDATION_PAIRED_URL` to point at hosted copies
of these files; otherwise both fall back to the local copies under `data/`.

## Catalog reference

What each catalog is, what wavelength/band it's built from, what kind of target it selects, and —
important for interpreting the plots below — whether the position used for matching is the
catalog's **own** astrometric measurement or a **Gaia DR3** position obtained by cross-identifying
the catalog's targets against Gaia.

**RFC and ICRF3 overlap substantially: ICRF3 is essentially a subset of RFC.** ICRF3 is the
formally-adopted set of defining/candidate/other sources for the celestial reference frame
(4536 sources); RFC is the larger, more frequently updated VLBI compilation (~22,800 sources) that
ICRF3 itself draws from. They are kept as separate catalogs/selections here (rather than merged)
so each can be labeled and assessed on its own — not because the underlying source lists are
independent.

| Catalog | Band | Targets / selection | Position used | Reference |
|---|---|---|---|---|
| **RFC** | Radio (VLBI, S/X band) | ~22,800 compact radio sources, mostly AGN cores | Catalog's own VLBI position (typically < 1 mas) | Petrov & Kovalev 2025, [ApJS 276, 38](https://doi.org/10.3847/1538-4365/ad8c36); data from [astrogeo.org/sol/rfc](https://astrogeo.org/sol/rfc/) |
| **ICRF3** | Radio (VLBI, S/X band) | 4536 defining/candidate/other sources of the celestial reference frame (AGN) — a subset of RFC | Catalog's own VLBI position | Charlot et al. 2020, [A&A 644, A159](https://doi.org/10.1051/0004-6361/202038368); [VizieR J/A+A/644/A159](https://vizier.u-strasbg.fr/viz-bin/VizieR?-source=J/A+A/644/A159) |
| **GaiaAGN** | Optical (Gaia DR3 astrometry/photometry) | ~2.2M confirmed AGN (`gaiadr3.agn_cross_id`), the sample underlying the Gaia celestial reference frame | Gaia DR3 position, used as-is (no proper-motion correction — extragalactic, no measurable PM) | Gaia DR3 archive table `agn_cross_id`; see the [Gaia DR3 archive table documentation](https://gea.esac.esa.int/archive/documentation/GDR3/) — the associated reference-frame paper is Gaia Collaboration, Klioner et al. 2022, A&A 667, A148 |
| **GaiaQSO** | Optical (Gaia DR3) | ~6.6M quasar candidates (`gaiadr3.qso_candidates`) classified from Gaia astrometry, photometry and colour; extends GaiaAGN fainter (G > 20) | Gaia DR3 position, as-is | Gaia DR3 archive table `qso_candidates`; see the [Gaia DR3 archive table documentation](https://gea.esac.esa.int/archive/documentation/GDR3/) |
| **Quaia** | Optical + mid-IR (Gaia DR3 + unWISE) | 755,850 quasars, G < 20.0, built from Gaia DR3 quasar candidates refined with unWISE infrared photometry | Gaia DR3 position, as provided in the Quaia catalog | Storey-Fisher et al. 2024, [ApJ 964, 69](https://doi.org/10.3847/1538-4357/ad1328); data at [zenodo.org/records/10403370](https://zenodo.org/records/10403370) |
| **MilliquasGaia** | Compiled multi-wavelength (spectroscopic confirmation), position from Gaia | Milliquas v8, restricted to spectroscopically-confirmed types Q/A/B/N (photometric candidates and radio/X-ray association candidates excluded), further required to have a Gaia DR3 match within 1.5" | Gaia DR3 position (the Gaia counterpart found within 1.5" of the Milliquas position) | Flesch 2023, Million Quasars (Milliquas) Catalog v8; [VizieR VII/294](https://vizier.u-strasbg.fr/viz-bin/VizieR?-source=VII/294) |
| **DESIV161** | Optical spectroscopy (redshift/type) + optical imaging (position) | DESI EDR spectra with `OType` QSO or GALAXY and `ZWARN == 0` (reliable redshift), ~4100 sq deg EDR footprint | **Not Gaia** — Legacy Survey DR9 imaging position (own catalog position, ≲0.1" accuracy, comparable to Gaia) | DESI Collaboration 2023; [VizieR V/161](https://vizier.u-strasbg.fr/viz-bin/VizieR?-source=V/161) |
| **GaiaVarStar** | Optical (Gaia DR3) | Rotation-modulation variable stars (`gaiadr3.vari_rotation_modulation`), RUWE < 1.4 | Gaia DR3 position, propagated from the DR3 reference epoch (J2016.0) to the observation epoch using the source's proper motion | Gaia DR3 archive table `vari_rotation_modulation`; see the [Gaia DR3 archive table documentation](https://gea.esac.esa.int/archive/documentation/GDR3/) |
| **Tycho2** *(baseline)* | Optical | 2.5M stars observed by Tycho aboard Hipparcos, down to ~11.5 mag | Catalog's own astrometric position, with proper motion | Høg et al. 2000; [VizieR I/259/tyc2](https://vizier.u-strasbg.fr/viz-bin/VizieR-3?-source=I/259/tyc2); [guide (PDF)](http://www.astro.ku.dk/~cf/CD/docs/guide.pdf) |

Only GaiaVarStar and Tycho2 apply a proper-motion/epoch correction — the radio and extragalactic
catalogs (RFC, ICRF3, GaiaAGN, GaiaQSO, Quaia, MilliquasGaia, DESIV161) describe sources with no
measurable proper motion, so their catalog positions are used as-is (see `docs/astromon.rst`).

**Note on the three Gaia-archive-table entries (GaiaAGN, GaiaQSO, GaiaVarStar):** these are ESA
Gaia DR3 archive tables rather than a single named external publication, so the citation above
points at the archive table documentation as the authoritative reference; the associated papers
are cited where I'm confident of the exact reference and should be double-checked against the
[Gaia DR3 documentation](https://gea.esac.esa.int/archive/documentation/GDR3/) before this goes
into anything citable.

**ICRF2**, the reference set used before ICRF3 superseded it, was fully retired from this
codebase (no getter function, not in `CROSS_MATCHES_ARGS`, absent from every DB backup checked)
and is not included below. See the note in the data section above.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from astropy import table

SAMPLE_DATA_URL = os.environ.get(
    "ASTROMON_VALIDATION_SAMPLE_URL",
    "data/catalog_match_validation_sample.ecsv",
)

matches = table.Table.read(SAMPLE_DATA_URL)

# Plot order matches cross_match._MATCH_HIERARCHY exactly: ICRF3 before RFC, since ICRF3
# is a subset of RFC and has to be tried first in a combined selection or it would never
# win a match (it would already have been claimed by RFC). Tycho2 and ICRF2 last -- both
# are "old baseline" catalogs, not new ones (ICRF2 is the pre-this-branch reference
# catalog, kept only for comparison -- see section 7).
CATALOG_ORDER = [
    "icrf3",
    "rfc",
    "gaia_agn",
    "quaia",
    "desi_v161",
    "milliquas_gaia",
    "gaia_qso",
    "gaia_var_star",
    "tycho2",
    "icrf2",
]
CATALOG_LABELS = {
    "icrf3": "ICRF3",
    "rfc": "RFC",
    "gaia_agn": "GaiaAGN",
    "quaia": "Quaia",
    "desi_v161": "DESIV161",
    "milliquas_gaia": "MilliquasGaia",
    "gaia_qso": "GaiaQSO",
    "gaia_var_star": "GaiaVarStar",
    "tycho2": "Tycho2 (old baseline)",
    "icrf2": "ICRF2 (old baseline)",
}

print(f"Loaded {len(matches)} rows from {SAMPLE_DATA_URL}")
matches[:5]

## 1. Overview: how many matches, and how tight are they

A healthy counterpart catalog should produce plenty of matches, with most of them landing well
inside the 3" match radius — a distribution piled up near dr=0 indicates real astrophysical
counterparts, while a distribution spread uniformly out to the cutoff would indicate confusion
with random field sources.

In [ ]:
DETECT_METHODS = ["celldetect", "gaussian_detect"]

summary_rows = []
for select_name in CATALOG_ORDER:
    for detect_method in DETECT_METHODS:
        sel = matches[
            (matches["select_name"] == select_name) & (matches["detect_method"] == detect_method)
        ]
        if len(sel) == 0:
            continue
        summary_rows.append(
            (
                CATALOG_LABELS[select_name],
                detect_method,
                len(sel),
                len(np.unique(sel["obsid"])),
                np.median(sel["dr"]),
                np.mean(sel["dr"] < 1.0) * 100,
                np.median(sel["snr"]),
                np.median(sel["r_angle"]),
            )
        )

summary = table.Table(
    rows=summary_rows,
    names=[
        "catalog",
        "detect_method",
        "n_matches (sample)",
        "n_obsids (sample)",
        "median dr [arcsec]",
        "pct dr<1\"",
        "median snr",
        "median r_angle [arcmin]",
    ],
)
for col in summary.colnames[4:]:
    summary[col].format = "%.2f"
summary

In [ ]:
n_cols = 3
n_rows = -(-len(CATALOG_ORDER) // n_cols)  # ceil division
fig, axes = plt.subplots(
    n_rows, n_cols, figsize=(13, 3.3 * n_rows), sharex=True, sharey=True
)
bins = np.linspace(0, 3, 31)
method_colors = {"celldetect": "steelblue", "gaussian_detect": "darkorange"}
for ax, select_name in zip(axes.flat, CATALOG_ORDER, strict=False):
    for detect_method in DETECT_METHODS:
        sel = matches[
            (matches["select_name"] == select_name) & (matches["detect_method"] == detect_method)
        ]
        ax.hist(
            sel["dr"],
            bins=bins,
            color=method_colors[detect_method],
            alpha=0.6,
            label=f"{detect_method} (n={len(sel)})",
        )
    ax.set_title(CATALOG_LABELS[select_name], fontsize=10)
for ax in axes.flat[len(CATALOG_ORDER) :]:
    ax.axis("off")
axes.flat[0].legend(fontsize=7)
for ax in axes[-1]:
    ax.set_xlabel("dr [arcsec]")
for ax in axes[:, 0]:
    ax.set_ylabel("count")
fig.suptitle(
    "Angular offset between X-ray source and catalog counterpart, by detect method", y=1.02
)
fig.tight_layout()

**What to look for:** every catalog above should show a peak well under 1", with a tail that
thins out before the 3" cutoff (the red dashed line is the median). A catalog whose histogram is
flat or piles up near the cutoff would be dominated by chance coincidences rather than real
counterparts.

## 2. Is match quality biased by X-ray SNR?

Calibration matches need to be reliable across the range of source brightness actually used, not
just for the brightest sources. If dr degrades sharply with lower SNR, that catalog's matches
should be weighted or cut accordingly.

celldetect and gaussian_detect SNR are **not on the same scale** (gaussian SNR runs roughly ~13x
celldetect SNR for the same source — the two tools use different noise definitions; see the
`snr=9.5` note by `CROSS_MATCHES_ARGS` in `astromon/cross_match.py`), so the two methods are
plotted separately rather than overlaid on one SNR axis.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
colors = plt.cm.tab10(np.linspace(0, 1, len(CATALOG_ORDER)))

for color, select_name in zip(colors, CATALOG_ORDER, strict=True):
    sel = matches[matches["select_name"] == select_name]
    ax.scatter(
        sel["snr"], sel["dr"], s=8, alpha=0.4, color=color, label=CATALOG_LABELS[select_name]
    )

ax.set_xscale("log")
ax.set_xlabel("X-ray source SNR")
ax.set_ylabel("dr [arcsec]")
ax.set_title("dr vs. SNR")
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()

## 3. Magnitude coverage

For the AGN/quasar catalogs in particular, it's worth checking whether match quality holds up at
the faint end, since that's where the sample depth (and therefore the astrometric leverage) comes
from.

In [ ]:
n_cols = 3
n_rows = -(-len(CATALOG_ORDER) // n_cols)  # ceil division
fig, axes = plt.subplots(n_rows, n_cols, figsize=(13, 3.3 * n_rows), sharey=True)
for ax, select_name in zip(axes.flat, CATALOG_ORDER, strict=False):
    sel = matches[matches["select_name"] == select_name]
    mag = np.asarray(sel["mag"])
    finite = np.isfinite(mag)
    if finite.sum() == 0:
        ax.set_title(f"{CATALOG_LABELS[select_name]} (no mag)", fontsize=10)
        continue
    ax.scatter(mag[finite], sel["dr"][finite], s=8, alpha=0.4, color="darkorange")
    ax.set_title(CATALOG_LABELS[select_name], fontsize=10)
for ax in axes.flat[len(CATALOG_ORDER) :]:
    ax.axis("off")
for ax in axes[-1]:
    ax.set_xlabel("mag")
for ax in axes[:, 0]:
    ax.set_ylabel("dr [arcsec]")
fig.suptitle("dr vs. counterpart magnitude", y=1.02)
fig.tight_layout()

## 4. Sky and time coverage

A catalog that only contributes matches in a handful of fields or a narrow time window is much
less useful for calibration than one that samples the sky and mission lifetime broadly.

In [ ]:
fig = plt.figure(figsize=(11, 6))
ax = fig.add_subplot(111, projection="aitoff")
for color, select_name in zip(colors, CATALOG_ORDER, strict=True):
    sel = matches[matches["select_name"] == select_name]
    ra_wrapped = np.radians(((np.asarray(sel["x_ra"]) + 180) % 360) - 180)
    dec_rad = np.radians(sel["x_dec"])
    ax.scatter(
        ra_wrapped, dec_rad, s=6, alpha=0.5, color=color, label=CATALOG_LABELS[select_name]
    )
ax.grid(True)
ax.set_title("Sky distribution of matched X-ray sources")
ax.legend(fontsize=8, ncol=3, loc="upper right", bbox_to_anchor=(1.15, 1.15))
fig.tight_layout()

### Systematic offset per catalog over time

`dy`/`dz` are the signed offsets (X-ray minus catalog position, in the SIM-Y/Z frame) rather than
the unsigned `dr`. A catalog with a real systematic problem (a residual proper-motion effect, an
epoch mismatch, a subtle position bias) would show up as a *cluster* of points sitting off zero
for a period, even if the `dr` scatter looks fine. Plotting raw points (not binned medians) avoids
the small-N noise a per-bin median can show when a bin has only a handful of matches -- worth
keeping in mind if a binned version of this plot is shown elsewhere and looks noisier than this.

**Rebased to a fixed CALALIGN epoch.** Plotting raw `dy`/`dz` over the mission would otherwise mix
real signal with step changes that happen purely because the CALALIGN aspect-alignment solution
was updated at various times -- a calibration artifact, not a catalog problem. `dy_rebased`/
`dz_rebased` (used below instead of `dy`/`dz`) undo whichever CALALIGN was in effect when each
observation was processed and re-express it relative to the latest alignment epoch, following the
recipe in Section 1 of `celmon-final-model.ipynb` in the sibling `absolute_astrometry` repo:
`dy_rebased = dy - (calalign_dy - ref_calalign_dy)`. See `rebase_dy_dz` in
`extract_catalog_validation_sample.py`.

**Caveat, checked directly rather than assumed:** the CALALIGN file cache used here (checked in 3
places on this machine, all identical) stops at 2021-07-02. Comparing `dy_rebased` to raw `dy` in
the sample, rebasing changes ~30% of pre-2021.5 rows (by up to ~0.4") but **0% of post-2021.5
rows** -- past that date there is no newer CALALIGN entry to rebase against, so `dy_rebased`
equals `dy` exactly there. Whether the CALALIGN cache is actually complete past 2021 (no real
update happened) or just stale on this machine is not confirmed either way -- worth checking
against the production CALDB mount before trusting the post-2021 points as fully rebased.

In [ ]:
from astropy.time import Time

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
for color, select_name in zip(colors, CATALOG_ORDER, strict=True):
    sel = matches[matches["select_name"] == select_name]
    years = Time(sel["tstart"], format="cxcsec").decimalyear
    if select_name == "tycho2":
        # Tycho2 is the baseline everything else is judged against -- give it a
        # marker that stands out from the crowd of same-size dots instead of
        # blending in as just another color.
        axes[0].scatter(
            years,
            sel["dy_rebased"],
            marker="x",
            s=40,
            linewidths=1.3,
            color="black",
            label=CATALOG_LABELS[select_name],
        )
        axes[1].scatter(
            years, sel["dz_rebased"], marker="x", s=40, linewidths=1.3, color="black"
        )
    else:
        axes[0].scatter(
            years,
            sel["dy_rebased"],
            s=8,
            alpha=0.4,
            color=color,
            label=CATALOG_LABELS[select_name],
        )
        axes[1].scatter(years, sel["dz_rebased"], s=8, alpha=0.4, color=color)

axes[0].axhline(0, color="gray", lw=0.8, ls=":")
axes[1].axhline(0, color="gray", lw=0.8, ls=":")
axes[0].set_ylabel("dy_rebased [arcsec]")
axes[1].set_ylabel("dz_rebased [arcsec]")
axes[1].set_xlabel("observation year")
axes[0].set_title(
    "X-ray minus catalog offset per catalog (rebased to latest CALALIGN), raw points"
    " -- Tycho2 baseline marked with black x's"
)
axes[0].legend(fontsize=8, ncol=3)
fig.tight_layout()

In [ ]:
from astropy.time import Time

fig, ax = plt.subplots(figsize=(11, 4))
for color, select_name in zip(colors, CATALOG_ORDER, strict=True):
    sel = matches[matches["select_name"] == select_name]
    years = Time(sel["tstart"], format="cxcsec").decimalyear
    ax.scatter(years, sel["dr"], s=8, alpha=0.4, color=color, label=CATALOG_LABELS[select_name])
ax.set_xlabel("observation year")
ax.set_ylabel("dr [arcsec]")
ax.set_title("Match quality over the mission lifetime")
ax.legend(fontsize=8, ncol=3)
fig.tight_layout()

## 5. Comparison to the Tycho2 baseline

Tycho2 has anchored `astromon_21`/`astromon_22` for years, so it's the natural yardstick: do the
new catalogs match at least as tightly?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
data = [matches[matches["select_name"] == name]["dr"] for name in CATALOG_ORDER]
bp = ax.boxplot(data, labels=[CATALOG_LABELS[n] for n in CATALOG_ORDER], showfliers=False)
ax.set_ylabel("dr [arcsec]")
ax.set_title("dr distribution by catalog (outliers hidden for scale)")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
fig.tight_layout()

## 6. celldetect vs. gaussian_detect: which centroids closer to the counterpart?

The plots above treat celldetect and gaussian_detect sources as separate populations. But most
catalog counterparts are actually matched independently by *both* detection methods in the same
obsid — the same physical X-ray source, centroided two different ways. That gives a direct, paired
comparison: for the same counterpart, is dr smaller with the celldetect centroid or the
gaussian_detect centroid?

This reads a second sample file, `catalog_match_paired_sample.ecsv` (see
`extract_paired_sample` in `extract_catalog_validation_sample.py`), where each row is one
counterpart with both `..._cell` and `..._gauss` columns rather than one row per detect method.

In [ ]:
PAIRED_DATA_URL = os.environ.get(
    "ASTROMON_VALIDATION_PAIRED_URL",
    "data/catalog_match_paired_sample.ecsv",
)
pairs = table.Table.read(PAIRED_DATA_URL)
print(f"Loaded {len(pairs)} celldetect/gaussian_detect pairs from {PAIRED_DATA_URL}")
pairs[:5]

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))
for color, select_name in zip(colors, CATALOG_ORDER, strict=True):
    sel = pairs[pairs["select_name"] == select_name]
    ax.scatter(
        sel["dr_cell"],
        sel["dr_gauss"],
        s=10,
        alpha=0.4,
        color=color,
        label=CATALOG_LABELS[select_name],
    )
lim = 3.0
ax.plot([0, lim], [0, lim], "k--", lw=1, label="equal dr")
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_xlabel("dr, celldetect centroid [arcsec]")
ax.set_ylabel("dr, gaussian_detect centroid [arcsec]")
ax.set_title("Same counterpart, two centroiding methods")
ax.legend(fontsize=7, ncol=2, loc="lower right")
ax.set_aspect("equal")
fig.tight_layout()

In [ ]:
from scipy.stats import wilcoxon

pair_rows = []
for select_name in [*CATALOG_ORDER, None]:
    sel = pairs if select_name is None else pairs[pairs["select_name"] == select_name]
    if len(sel) < 2:
        continue
    diff = sel["dr_gauss"] - sel["dr_cell"]  # negative => gaussian_detect centroids closer
    # Wilcoxon signed-rank test on the paired dr difference: is the median difference
    # from zero more than expected by chance, given the whole distribution of differences?
    stat = wilcoxon(diff)
    pair_rows.append(
        (
            "ALL" if select_name is None else CATALOG_LABELS[select_name],
            len(sel),
            float(np.mean(diff < 0) * 100),
            float(np.median(diff)),
            float(stat.pvalue),
        )
    )

pair_summary = table.Table(
    rows=pair_rows,
    names=[
        "catalog",
        "n_pairs (sample)",
        "pct gaussian closer",
        "median (dr_gauss - dr_cell)",
        "wilcoxon p-value",
    ],
)
pair_summary["pct gaussian closer"].format = "%.1f"
pair_summary["median (dr_gauss - dr_cell)"].format = "%.4f"
pair_summary["wilcoxon p-value"].format = "%.2e"
pair_summary

**Reading this table:** "pct gaussian closer" > 50% and a negative median difference both say
gaussian_detect centroids land closer to the counterpart more often than not; the Wilcoxon
p-value tests whether that paired difference is distinguishable from zero (not whether the *size*
of the improvement is practically important — check the median difference and the scatter plot
above for that). A catalog where gaussian_detect is *not* better (pct closer near or below 50%,
p-value not small) is a real counter-example worth calling out rather than smoothing over.

## 7. How many new matches do these catalogs actually get us?

Everything above checks *quality* -- are the new catalogs' matches good ones? This section
checks *quantity*: how many X-ray sources get an astrometric counterpart today that had
**none at all** under the pre-this-branch catalog set?

"Old baseline" here is **ICRF2 union Tycho2** -- the actual historical `astromon_21`
catalog set (`catalogs=["ICRS", "Tycho2"]`, where "ICRS" was this codebase's old name for
ICRF2; see `docs/quick.rst`). ICRF2 was fully retired when ICRF3/RFC replaced it, so it was
added back (`astromon/cross_match.py`, `astromon/scripts/backfill_icrf2.py`) purely to serve
as this comparison point -- it is not part of any combined hierarchy and needed no
reprocessing of `astromon_23`-`27`.

"New" is `astromon_23`: the combined hierarchy across every catalog added in this branch
(RFC, ICRF3, GaiaAGN, Quaia, DESIV161, MilliquasGaia, GaiaQSO, GaiaVarStar) plus Tycho2.

A source counts as "gained" only if it has **no** counterpart at all under the old baseline
-- not merely a different one -- so this measures genuinely new astrometric coverage, not a
catalog label changing for a source that was already usable. Computed against the **full
DB** (`compute_match_gains_summary` in `extract_catalog_validation_sample.py`), not the
capped sample above -- these are small summary counts, so there's no reason to sample them.

In [ ]:
GAINS_DATA_URL = os.environ.get(
    "ASTROMON_VALIDATION_GAINS_URL",
    "data/catalog_match_gains_summary.ecsv",
)
gains = table.Table.read(GAINS_DATA_URL)
gains

In [ ]:
per_catalog = gains[gains["catalog"] != "TOTAL"]
total_gained = int(gains["n_gained"][gains["catalog"] == "TOTAL"][0])
n_new_total = int(gains["n_new_total"][0])

fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = plt.cm.tab10(np.linspace(0, 1, len(per_catalog)))
ax.bar(per_catalog["catalog"], per_catalog["n_gained"], color=bar_colors)
ax.set_ylabel("newly-gained X-ray sources")
ax.set_title(
    f"Sources with no counterpart under the old baseline, by catalog\n"
    f"({total_gained:,} of {n_new_total:,} astromon_23 matches, "
    f"{100 * total_gained / n_new_total:.0f}%, are genuinely new)"
)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
fig.tight_layout()

## Conclusion

Fill in after reviewing the plots above for the current DB snapshot, e.g.:

- All eight new catalogs produce matches with median dr comparable to (or tighter than) the
  Tycho2 baseline, consistent with the sub-mas to sub-arcsec catalog position accuracy quoted in
  `docs/astromon.rst`.
- No strong dr degradation with SNR or off-axis angle within the ranges sampled, so no additional
  per-catalog cuts appear necessary beyond what `CROSS_MATCHES_ARGS` already applies.
- Sky and time coverage [summarize once plots are reviewed against the full DB, not just this
  capped sample].

**Caveat:** this notebook runs against a capped sample (400 matches/catalog) for portability. For
any conclusion sensitive to rare outliers or full-sample statistics, rerun
`extract_catalog_validation_sample.py` with a larger `--n-per-catalog`, or query
`astromon.get_cross_matches` directly against the full DB.